<a href="https://colab.research.google.com/github/hazami-razip/Machine-Learning/blob/main/Assignment1_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Question 1 (20 Marks) **

**A) Select the independent variables that you believe are most suitable for predicting median_house_value.**

Answer:
*   median_income: Higher income areas correlate with higher property costs.
*   ocean_proximity: Location relative to the coast significantly impacts demand and price.
*   housing_median_age: Older homes may have historical value or lower prices due to maintenance, while newer homes command premiums.
*   latitude & longitude: Real estate is location-dependent; specific coordinates capture regional market trends.
*   total_rooms & population: These provide a sense of the scale and density of the neighborhood.



**Import the datasets given**

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [5]:
dataset = pd.read_csv('housing.csv')
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values

In [7]:
print(X)

[[-1.2223e+02  3.7880e+01  4.1000e+01 ...  1.2600e+02  8.3252e+00
   4.5260e+05]
 [-1.2222e+02  3.7860e+01  2.1000e+01 ...  1.1380e+03  8.3014e+00
   3.5850e+05]
 [-1.2224e+02  3.7850e+01  5.2000e+01 ...  1.7700e+02  7.2574e+00
   3.5210e+05]
 ...
 [-1.2122e+02  3.9430e+01  1.7000e+01 ...  4.3300e+02  1.7000e+00
   9.2300e+04]
 [-1.2132e+02  3.9430e+01  1.8000e+01 ...  3.4900e+02  1.8672e+00
   8.4700e+04]
 [-1.2124e+02  3.9370e+01  1.6000e+01 ...  5.3000e+02  2.3886e+00
   8.9400e+04]]


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

**B) Handling Missing Values:** The total_bedrooms column often contains null entries in this dataset. We use Median Imputation to fill these without introducing the bias of extreme outliers.

**Categorical Encoding:** ocean_proximity is text-based. We use One-Hot Encoding to convert it into binary columns so the models can process the data mathematically.

**Data Splitting:** We split the data into 80% Training and 20% Testing sets to evaluate generalization.

In [9]:
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing(as_frame=True)
df = data.frame

# Define features and target
X = df.drop("MedHouseVal", axis=1) # median_house_value
y = df["MedHouseVal"]

# Preprocessing Pipeline
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data Prepared and Split Successfully.")

Data Prepared and Split Successfully.


**C) Linear Regression**

In [10]:
from sklearn.linear_model import LinearRegression

# Build and train
lr_model = Pipeline(steps=[('preprocessor', preprocessor),
                           ('regressor', LinearRegression())])
lr_model.fit(X_train, y_train)

# Predict and Evaluate
y_pred_lr = lr_model.predict(X_test)

print("--- Multiple Linear Regression Results ---")
print(f"R² Score: {r2_score(y_test, y_pred_lr):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred_lr):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lr)):.4f}")

--- Multiple Linear Regression Results ---
R² Score: 0.5758
MAE: 0.5332
RMSE: 0.7456


**D) Polynomial Regression**:

We use a degree of 2. A degree of 2 captures non-linear interactions between variables (like income and location) without the extreme risk of overfitting or high computational cost associated with higher degrees.

In [11]:
from sklearn.preprocessing import PolynomialFeatures

# Build and train
poly_model = Pipeline(steps=[('preprocessor', preprocessor),
                             ('poly', PolynomialFeatures(degree=2)),
                             ('regressor', LinearRegression())])
poly_model.fit(X_train, y_train)

# Predict and Evaluate
y_pred_poly = poly_model.predict(X_test)

print("--- Polynomial Regression (Degree 2) Results ---")
print(f"R² Score: {r2_score(y_test, y_pred_poly):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred_poly):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_poly)):.4f}")

--- Polynomial Regression (Degree 2) Results ---
R² Score: 0.6457
MAE: 0.4670
RMSE: 0.6814


**E) Support Vector Regression**

We use the RBF (Radial Basis Function) kernel. This is chosen because house prices rarely follow a linear pattern, and the RBF kernel is excellent at finding complex, non-linear boundaries in higher-dimensional space.

In [13]:
from sklearn.svm import SVR

# Build and train
svr_model = Pipeline(steps=[('preprocessor', preprocessor),
                            ('regressor', SVR(kernel='rbf', C=1.0, epsilon=0.1))])
svr_model.fit(X_train, y_train)

# Predict and Evaluate
y_pred_svr = svr_model.predict(X_test)

print("--- Support Vector Regression (RBF) Results ---")
print(f"R² Score: {r2_score(y_test, y_pred_svr):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred_svr):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_svr)):.4f}")

--- Support Vector Regression (RBF) Results ---
R² Score: 0.7276
MAE: 0.3986
RMSE: 0.5975
